# 11 — Resultados finais
Comparação no conjunto de teste: baselines × MLP, com IC 95% por bootstrap para a AUC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve, balanced_accuracy_score, accuracy_score,
                             f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay)
import config

yte = np.load(config.ARQ_DATASET, allow_pickle=True)["y_test"]
probas = dict(np.load(config.RESULTADOS / "proba_baselines.npz"))
probas["MLP"] = np.load(config.RESULTADOS / "proba_mlp.npy")
print({k: v.shape for k, v in probas.items()}, yte.shape)

In [ ]:
rng = np.random.default_rng(config.SEED)

def auc_ic(y, p, n=1000):
    aucs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        aucs.append(roc_auc_score(y[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

linhas = []
for nome, p in probas.items():
    pred = (p >= 0.5).astype(int)
    lo, hi = auc_ic(yte, p)
    linhas.append({"modelo": nome, "AUC": roc_auc_score(yte, p), "AUC_IC95": f"[{lo:.3f}, {hi:.3f}]",
                   "Acc_balanceada": balanced_accuracy_score(yte, pred), "Acurácia": accuracy_score(yte, pred),
                   "Precisão": precision_score(yte, pred, zero_division=0),
                   "Sensibilidade": recall_score(yte, pred),
                   "Especificidade": recall_score(yte, pred, pos_label=0), "F1": f1_score(yte, pred)})
tab = pd.DataFrame(linhas).sort_values("AUC", ascending=False).round(3)
display(tab)
tab.to_csv(config.RESULTADOS / "resultados_finais.csv", index=False)

In [ ]:
plt.figure(figsize=(6, 6))
for nome, p in probas.items():
    fpr, tpr, _ = roc_curve(yte, p)
    plt.plot(fpr, tpr, label=f"{nome} (AUC {roc_auc_score(yte, p):.2f})")
plt.plot([0, 1], [0, 1], "k--", alpha=.4)
plt.xlabel("1 − Especificidade"); plt.ylabel("Sensibilidade"); plt.legend(); plt.title("Curvas ROC (teste)")
plt.tight_layout(); plt.savefig(config.FIGURAS / "11_roc.png", dpi=150); plt.show()

In [ ]:
n = len(probas)
fig, ax = plt.subplots(1, n, figsize=(3.6 * n, 3.6))
for a, (nome, p) in zip(np.atleast_1d(ax), probas.items()):
    ConfusionMatrixDisplay(confusion_matrix(yte, (p >= 0.5).astype(int)),
                           display_labels=["Controle", "TDAH"]).plot(ax=a, colorbar=False)
    a.set_title(nome)
plt.tight_layout(); plt.savefig(config.FIGURAS / "11_matrizes_confusao.png", dpi=150); plt.show()

Resultados em `resultados/resultados_finais.csv`; figuras em `resultados/figuras/`.